# Chapter 5 &mdash; The DFA Markdown Syntax, by Worked Example

**Concept 13 of the Chapter 5 decomposition:** *The DFA Markdown Syntax, by Worked Example*

`DFA` header, then `State : In -> ToState !! comment`, with `|` for alternatives.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter5-DFADsg/Concept-DFA-Markdown-Syntax/Concept-DFA-Markdown-Syntax.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.AnimateDFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateDFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateDFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


The full DFA syntax, in four rules:

* line 1 is the word **`DFA`**;
* each transition is **`State : In -> ToState`**;
* **`|`** offers alternative inputs: `Q : 0 | 1 -> R`;
* **`!!`** begins a comment that runs to end of line.

State names carry the structure: leading **`I`** initial, **`F`** final, **`IF`** both.
Everything after those letters is yours, so use it.

Whitespace is free, so align the arrows and the machine reads like a table.

## 2. Definitions

### Every feature of the syntax, in one machine

In [ ]:
D = md2mc('''DFA
!! ------------------------------------------------------------------
!! L = strings over {0,1,2} that contain at least one 2
!! ------------------------------------------------------------------
I : 0 | 1 -> I          !! '|' gives alternatives on one line
I : 2     -> F          !! seen a 2 -- we are done
F : 0 | 1 | 2 -> F      !! F is a trap, in the good sense
''')

### Common syntax mistakes, and what they do

In [ ]:
def try_parse(src):
    try:
        D = md2mc(src); return "ok: states " + str(sorted(D["Q"]))
    except Exception as e:
        return "%s: %s" % (type(e).__name__, str(e)[:70])

<!-- nav-strip -->

---

&larr;&nbsp;[Ch5&nbsp;12.&nbsp;Automd: a Markdown Language for All Machines](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter5-DFADsg/Concept-Automd-Markdown/Concept-Automd-Markdown.ipynb) &nbsp;&middot;&nbsp; [**Chapter 5** index](https://github.com/ganeshutah/Jove/blob/master/Chapter5-DFADsg/README.md) &nbsp;&middot;&nbsp; [Ch5&nbsp;14.&nbsp;Running and Testing DFA in Jove with `nthnumeric`](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter5-DFADsg/Concept-Testing-With-Nthnumeric/Concept-Testing-With-Nthnumeric.ipynb)&nbsp;&rarr;

---

## 3. Tests

The alternatives expand to separate entries in $\delta$.

In [ ]:
for kv in sorted(D["Delta"].items()): print("  ", kv)
print("\n|Delta| =", len(D["Delta"]), " = |Q| x |Sigma| =",
      len(D["Q"]) * len(D["Sigma"]))
assert len(D["Delta"]) == len(D["Q"]) * len(D["Sigma"])

And it recognises what the comment claims.

In [ ]:
for s in ['', '0', '2', '0102', '1111', '20']:
    print("  %-7r contains a 2? %-5s dfa says %s" % (s, '2' in s, accepts_dfa(D, s)))
from itertools import product
assert all(accepts_dfa(D, ''.join(p)) == ('2' in ''.join(p))
           for k in range(6) for p in product('012', repeat=k))

Two error cases worth recognising on sight.

In [ ]:
print("two initial states:")
print("   ", try_parse('DFA\nI1 : 0 -> I2\nI2 : 0 -> I1\n'))
print("\nno final state (parses fine, accepts nothing):")
E = md2mc('DFA\nI : 0 -> A\nA : 0 -> I\n')
print("    F =", E["F"], " accepts '00'?", accepts_dfa(E, '00'))
assert E["F"] == set()
print("\nThe second is the dangerous one -- it is SILENT.")

## 4. Animation

The `|` shorthand drawn out: Jove fuses parallel edges when FuseEdges=True.

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(D, FuseEdges=True)

## 5. Exercises


1. Write the even-0s machine using `|` wherever possible.
2. What does `md2mc` do with a transition from an undeclared state?
3. Why is "no final state" more dangerous than "two initial states"?

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter5-DFADsg/Concept-DFA-Markdown-Syntax')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')